## This is the code to train the model and acquire influence and loss for Sensitivity to Outliers Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 16000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is almost the same code as the base code. Please refer to the base code for more detailed explanation.** The differences here are: 1. Train_sizes are limited to 5 choices. 2. The loss of the training points are stored as a file for later analysis.

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> Store the Loss Lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last three block3 to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [47]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [48]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [49]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [50]:
import random
from keras.optimizers import SGD

In [51]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [52]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 16000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**If you just want to use our setting to test, the only thing you need to change here is the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to multi-verify the experiement results.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [53]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
n_features=160
seed=42
sep = 5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [54]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [55]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     -11.782908  -8.132727  16.152943  -9.137928   4.798538 -31.164432   
1       4.337057  -3.322490  -5.760765  -4.599484  13.619708  -0.447431   
2      -6.162488  10.509963   8.350375   3.877896  10.781976  -3.668239   
3       1.279108  -2.215843   5.809480  -6.197446   4.975266  -9.387154   
4       2.271039  -3.547205   1.348749 -10.493556  10.239804  -7.990558   
...          ...        ...        ...        ...        ...        ...   
16495  -6.716227  -3.941982   6.511306  -1.178930 -10.376654  -9.079567   
16496   4.277247 -15.312607  10.519820  -0.178716  -3.619789 -10.264994   
16497  -8.898158   2.233196  -6.868816  -1.021291   9.341948  15.122732   
16498  -3.610179 -17.301590   2.750310   6.510539  -0.240444  -2.099331   
16499 -17.536142   5.698689   9.265016   3.931781   6.659502  10.521925   

       feature_7  feature_8  feature_9  feature_10  ...  feature_153  \
0       8.887783  -4.063796

In [56]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [57]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0 -11.782908  -8.132727  16.152943  -9.137928   4.798538 -31.164432   
1   4.337057  -3.322490  -5.760765  -4.599484  13.619708  -0.447431   
2  -6.162488  10.509963   8.350375   3.877896  10.781976  -3.668239   
3   1.279108  -2.215843   5.809480  -6.197446   4.975266  -9.387154   
4   2.271039  -3.547205   1.348749 -10.493556  10.239804  -7.990558   

   feature_7  feature_8  feature_9  feature_10  ...  feature_153  feature_154  \
0   8.887783  -4.063796  21.086774   -1.520991  ...     7.819484    -8.721757   
1  -2.525553   9.558629   7.793142    4.989492  ...     0.182679    -0.204365   
2 -17.248069   6.155376   3.674610    8.811081  ...     7.133064    16.611964   
3 -16.208869   2.459683  20.384366   11.957595  ...     6.951633     2.328660   
4 -16.237140  15.315085  12.821372   -1.751887  ...     4.459658     5.218489   

   feature_155  feature_156  feature_157  feature_158  feature_159  \
0    -0.265044  

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [58]:
features_to_test = 10
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

Again, change here if you wish to change the number of train size

In [59]:
train_df = nested_train_dfs[9]

In [60]:
train_df["label"].value_counts()

label
1    5025
0    4975
Name: count, dtype: int64

In [61]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train.shape)

(10000, 11)


In [62]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)

(500, 11)


In [63]:
test_df["label"].value_counts()

label
0    255
1    245
Name: count, dtype: int64

5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [64]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

# D = pairwise_distances(X_all) 

In [65]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [66]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [67]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

**The only difference here is the loss for the training samples are stored and this shall be saved in a file later**

In [68]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [69]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

40/40 - 1s - loss: 0.8446 - accuracy: 0.4925 - val_loss: 0.7338 - val_accuracy: 0.5440 - 1s/epoch - 29ms/step
40/40 - 0s - loss: 0.6577 - accuracy: 0.6310 - val_loss: 0.6077 - val_accuracy: 0.6940 - 79ms/epoch - 2ms/step
40/40 - 0s - loss: 0.5754 - accuracy: 0.7271 - val_loss: 0.5418 - val_accuracy: 0.7700 - 73ms/epoch - 2ms/step
40/40 - 0s - loss: 0.5236 - accuracy: 0.7759 - val_loss: 0.4939 - val_accuracy: 0.8140 - 81ms/epoch - 2ms/step
40/40 - 0s - loss: 0.4839 - accuracy: 0.8028 - val_loss: 0.4562 - val_accuracy: 0.8240 - 75ms/epoch - 2ms/step
40/40 - 0s - loss: 0.4516 - accuracy: 0.8218 - val_loss: 0.4253 - val_accuracy: 0.8440 - 74ms/epoch - 2ms/step
40/40 - 0s - loss: 0.4249 - accuracy: 0.8356 - val_loss: 0.4004 - val_accuracy: 0.8600 - 75ms/epoch - 2ms/step
40/40 - 0s - loss: 0.4033 - accuracy: 0.8449 - val_loss: 0.3804 - val_accuracy: 0.8660 - 79ms/epoch - 2ms/step
40/40 - 0s - loss: 0.3856 - accuracy: 0.8519 - val_loss: 0.3636 - val_accuracy: 0.8680 - 76ms/epoch - 2ms/step
40

2. Store the loss for each training sample here. Important for this experiment

In [70]:
train_loss_values = unreduced_loss_fn(y_train, model.predict(X_train)).numpy()

313/313 [==============================] - 0s 794us/step


# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC. One Loss list. 
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [71]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [72]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0            1  0.027767
1            2  0.014881
2            3  0.004174
3            4  0.002700
4            5  0.002164
...        ...       ...
9995      9996  0.030038
9996      9997  0.031931
9997      9998  0.000715
9998      9999  0.017813
9999     10000  0.022952

[10000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [73]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID         Score
0            1  5.376713e-04
1            2  9.908415e-05
2            3  1.347251e-05
3            4  1.314285e-05
4            5  1.143101e-05
...        ...           ...
9995      9996 -9.517032e-05
9996      9997 -1.563515e-05
9997      9998 -1.301336e-07
9998      9999  7.236763e-05
9999     10000 -7.840512e-06

[10000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing. Also, we store the previous loss list here.

In [74]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [75]:
TracIn_sorted.to_csv("TC_Train_Set_1.csv",index = False)
df_sorted.to_csv("IF_Train_Set_1.csv",index = False)

In [76]:
df_train_loss = pd.DataFrame({
    'Train_ID': train_ids,
    'Loss': train_loss_values
}).sort_values('Loss', ascending=False).reset_index(drop=True)
print(df_train_loss)

      Train_ID      Loss
0         6972  6.363977
1         2957  6.356202
2         6845  5.963240
3          382  5.693455
4         7194  5.630051
...        ...       ...
9995      1374  0.000042
9996      9506  0.000040
9997      2528  0.000025
9998      6211  0.000025
9999      3372  0.000005

[10000 rows x 2 columns]


In [77]:
df_train_loss.to_csv("Train_Loss_Train_Set_1.csv",index = False)